## Project 5: Statistical Decision Support

In [21]:
# Framework: Validating Business Assumptions with Empirical Testing

In [22]:
import pandas as pd
from scipy import stats

df = pd.read_csv("marketing_AB.csv")
df = df.rename(columns=lambda x: x.strip().replace(" ", "_"))

unique_users = df['user_id'].nunique()
total_rows = len(df)
print("Unique Users:", unique_users)
print("Total Rows:", total_rows)

group_counts = df['test_group'].value_counts()
print("\nSample Size Per Group:")
print(group_counts)

contingency = pd.crosstab(df['test_group'], df['converted'])
chi2_stat, p_val, dof, expected_freq = stats.chi2_contingency(contingency)

print("\nExpected Frequencies:")
print(expected_freq)

if (expected_freq >= 5).all() and unique_users == total_rows:
    print("\nStatistical assumptions validated. Proceeding with hypothesis testing.")
else:
    print("\nWarning: Statistical assumptions not fully met.")

Unique Users: 588101
Total Rows: 588101

Sample Size Per Group:
test_group
ad     564577
psa     23524
Name: count, dtype: int64

Expected Frequencies:
[[550327.71899045  14249.28100955]
 [ 22930.28100955    593.71899045]]

Statistical assumptions validated. Proceeding with hypothesis testing.


In [23]:
import numpy as np

In [24]:
#Setting up  Group Counts
contingency = pd.crosstab(df['test_group'], df['converted'])
contingency

converted,False,True
test_group,,
ad,550154,14423
psa,23104,420


## Applying statistical tests
Chi-Square Test and 
Two-Proportion Z-Test

In [25]:
from scipy import stats

chi2_stat, p_val_chi2, dof, expected = stats.chi2_contingency(contingency)

print("Chi-Square Stat:", round(chi2_stat, 2))
print("p-value:", p_val_chi2)

Chi-Square Stat: 54.01
p-value: 1.9989623063390075e-13


In [26]:
import numpy as np

n_ad = len(df[df['test_group'] == 'ad'])
n_psa = len(df[df['test_group'] == 'psa'])

p_ad = contingency.loc['ad', True] / n_ad
p_psa = contingency.loc['psa', True] / n_psa
p_pool = (contingency.loc['ad', True] + contingency.loc['psa', True]) / (n_ad + n_psa)

se = np.sqrt(p_pool * (1 - p_pool) * (1/n_ad + 1/n_psa))
z_stat = (p_ad - p_psa) / se

print("Z-Stat:", round(z_stat, 2))

Z-Stat: 7.37


In [27]:
#Two-Sample T-Test (Comparing Average Ads Seen)

converted_ads = df[df['converted'] == True]['total_ads']
not_converted_ads = df[df['converted'] == False]['total_ads']

t_stat, p_val_t = stats.ttest_ind(converted_ads, not_converted_ads, equal_var=False)

print("T-Stat:", round(t_stat, 2))
print("p-value:", p_val_t)

T-Stat: 84.18
p-value: 0.0


## Frame a business hypothesis

In [28]:
#Defining the Core Business Question & Metrics

In [29]:
ad_conv = df[df['test_group'] == 'ad']['converted'].mean()
psa_conv = df[df['test_group'] == 'psa']['converted'].mean()

print(f"Ad Conversion Rate:  {ad_conv * 100:.2f}%")
print(f"PSA Conversion Rate: {psa_conv * 100:.2f}%")

Ad Conversion Rate:  2.55%
PSA Conversion Rate: 1.79%


In [30]:
#Formulating Statistical Hypotheses (H0 and H1)

In [31]:
# H0 (Null Hypothesis): The ad campaign has NO impact on conversion rate (p_ad = p_psa).
# H1 (Alternative Hypothesis): The ad campaign INCREASES the conversion rate (p_ad > p_psa).

null_hypothesis = "Ad campaign shows no significant effect on conversions."
alt_hypothesis = "Ad campaign significantly increases user conversion rate."

print("H0:", null_hypothesis)
print("H1:", alt_hypothesis)

H0: Ad campaign shows no significant effect on conversions.
H1: Ad campaign significantly increases user conversion rate.


In [32]:
#Settin up Decision Criteria

In [33]:
alpha = 0.05  # 5% significance threshold

print(f"Significance Level (Alpha): {alpha}")
print(f"Decision Rule: Reject H0 if p-value < {alpha}")

Significance Level (Alpha): 0.05
Decision Rule: Reject H0 if p-value < 0.05


## Interpret results

In [34]:
# Evaluate p-value against Alpha

alpha = 0.05

print("STATISTICAL DECISION")
print("P-Value:", p_val_chi2)

if p_val_chi2 < alpha:
    print("Result: Statistically Significant (p < 0.05)")
    print("Decision: Reject Null Hypothesis (H0)")
else:
    print("Result: Not Statistically Significant (p >= 0.05)")
    print("Decision: Fail to Reject Null Hypothesis (H0)")

STATISTICAL DECISION
P-Value: 1.9989623063390075e-13
Result: Statistically Significant (p < 0.05)
Decision: Reject Null Hypothesis (H0)


In [35]:
# Calculate Business Impact & Uplift

p_ad = contingency.loc['ad', True] / len(df[df['test_group'] == 'ad'])
p_psa = contingency.loc['psa', True] / len(df[df['test_group'] == 'psa'])

lift = ((p_ad - p_psa) / p_psa) * 100

print("BUSINESS IMPACT")
print(f"Control (PSA) Conversion Rate: {p_psa*100:.2f}%")
print(f"Treatment (Ad) Conversion Rate: {p_ad*100:.2f}%")
print(f"Relative Conversion Lift:       +{lift:.2f}%")

BUSINESS IMPACT
Control (PSA) Conversion Rate: 1.79%
Treatment (Ad) Conversion Rate: 2.55%
Relative Conversion Lift:       +43.09%


In [36]:
# Executive Conclusion & Action

print("FINAL BUSINESS RECOMMENDATION")
print("1. Key Conclusion: The ad campaign drove a statistically significant uplift in conversion rate.")
print("2. Action: Scale up the ad campaign and allocate full marketing budget toward it.")

FINAL BUSINESS RECOMMENDATION
1. Key Conclusion: The ad campaign drove a statistically significant uplift in conversion rate.
2. Action: Scale up the ad campaign and allocate full marketing budget toward it.
